# Drift Detection — 04: Hidden Markov Models (HMM)

Explicitly models latent regimes the user transitions between. The model learns:
1. **Emission distributions:** what each regime's alignment scores look like
2. **Transition matrix:** probability of switching between regimes at each step

The signal taxonomy **emerges from the learned transition patterns** — crash = aligned→drifting,
fade = aligned→struggling→drifting, spike = aligned→drifting→aligned. No hand-coding needed.

**Doc reference:** `docs/evolution/drift_detection.md § 3.4`

### States (K = 3)

| State | Label | Expected scores |
|---|---|---|
| 0 | Aligned | Centered on +1, low variance |
| 1 | Struggling | Centered on 0, moderate variance |
| 2 | Drifting | Centered on -1, low-moderate variance |

### Implementation

Uses `statsmodels.tsa.regime_switching.MarkovRegression` — a Markov-switching regression
with K regimes fitted by EM (Hamilton filter).

### Data note

With ~10-15 steps per persona, per-user EM will often not converge reliably.
This notebook pools across all personas to estimate regime parameters, then decodes
individual trajectories using the pooled model.


In [ ]:
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns

sns.set_theme(style="whitegrid", font_scale=0.9)

from statsmodels.tsa.regime_switching.markov_regression import MarkovRegression
import warnings
warnings.filterwarnings('ignore')

# --- Load judge labels (204 personas, 1651 entries, integer {-1, 0, 1} scores) ---
# Using judge labels rather than human annotations: more data (all 204 personas vs 24),
# and clean integer scores (no averaging artefacts).

VALUE_COLS = [
    "alignment_self_direction", "alignment_stimulation", "alignment_hedonism",
    "alignment_achievement", "alignment_power", "alignment_security",
    "alignment_conformity", "alignment_tradition", "alignment_benevolence",
    "alignment_universalism",
]
SHORT_NAMES = [c.replace("alignment_", "") for c in VALUE_COLS]
DIM_LABELS  = ["SD", "ST", "HE", "AC", "PO", "SE", "CO", "TR", "BE", "UN"]

judge_df = pl.read_parquet("../../logs/judge_labels/judge_labels.parquet")
mean_df  = (
    judge_df
    .select(["persona_id", "t_index"] + VALUE_COLS)
    .with_columns([pl.col(c).cast(pl.Float64) for c in VALUE_COLS])
    .sort(["persona_id", "t_index"])
)

registry   = pl.read_parquet("../../logs/registry/personas.parquet").select(
                ["persona_id", "name", "core_values"])
id_to_name = dict(zip(registry["persona_id"].to_list(), registry["name"].to_list()))
id_to_core = dict(zip(registry["persona_id"].to_list(), registry["core_values"].to_list()))

persona_ids = sorted(mean_df["persona_id"].unique().to_list())
print(f"Loaded {len(judge_df)} judge-labelled entries across {len(persona_ids)} personas")

# --- Sample: verify input scores look correct ---
# Pick first persona with ≥5 entries so the sample is representative
sample_pid = next(
    pid for pid in persona_ids
    if mean_df.filter(pl.col("persona_id") == pid).height >= 5
)
sample_data = mean_df.filter(pl.col("persona_id") == sample_pid).head(5)
print(f"\nSample scores (first 5 entries) for '{id_to_name.get(sample_pid, sample_pid)}':")
print(f"  Core values: {id_to_core.get(sample_pid, [])}")
print(sample_data.select(["t_index"] + VALUE_COLS))
print("  Scores are integers in {{-1, 0, +1}} across all 10 Schwartz dimensions")


def get_persona_matrix(pid: str) -> tuple[list[int], np.ndarray]:
    pdata = mean_df.filter(pl.col("persona_id") == pid).sort("t_index")
    return pdata["t_index"].to_list(), np.array([pdata[c].to_list() for c in VALUE_COLS]).T


def get_profile_weights(pid: str) -> np.ndarray:
    core = id_to_core.get(pid, [])
    name_to_idx = {s.lower().replace("-", "_"): i for i, s in enumerate(SHORT_NAMES)}
    w = np.zeros(10)
    for v in core:
        key = v.lower().replace("-", "_").replace(" ", "_")
        if key in name_to_idx:
            w[name_to_idx[key]] = 1.0
    if w.sum() > 0:
        w /= w.sum()
    return w


def core_dim_indices(weights: np.ndarray, w_min: float = 0.15) -> list[int]:
    return [j for j, wj in enumerate(weights) if wj >= w_min]


## Pooled HMM Training

Concatenate all core-dimension alignment scores across personas and fit a 3-state Markov switching model.
This gives stable regime parameters despite the small per-persona sample size.

In [ ]:
W_MIN = 0.15
K     = 3      # number of latent states

# Collect all core-dimension score sequences
all_scores = []
for pid in persona_ids:
    t_idx, matrix = get_persona_matrix(pid)
    if len(t_idx) < 5:
        continue
    w = get_profile_weights(pid)
    for j in range(10):
        if w[j] >= W_MIN:
            all_scores.extend(matrix[:, j].tolist())

pooled = np.array(all_scores)
print(f"Pooled {len(pooled)} observations from core dimensions across all personas")
print(f"Mean: {pooled.mean():.3f}  Std: {pooled.std():.3f}  Range: [{pooled.min():.1f}, {pooled.max():.1f}]")

# Fit Markov Switching Model
# order=0 means intercept-only (no autoregressive terms)
model = MarkovRegression(pooled, k_regimes=K, order=0, switching_variance=True)
try:
    fit = model.fit(disp=False, maxiter=200)
    print(f"\nConverged: {fit.mle_retvals['converged']}")
    print(f"Log-likelihood: {fit.llf:.2f}")
    print(f"\nRegime means (intercepts):")
    for k in range(K):
        print(f"  State {k}: mean={fit.params[f'const[{k}]']:.3f}  "
              f"var={fit.params.get(f'sigma2[{k}]', fit.params.get('sigma2', 0)):.3f}")
    print(f"\nTransition matrix:")
    print(fit.regime_transition)
    POOLED_FIT = fit
except Exception as e:
    print(f"Fit failed: {e}")
    POOLED_FIT = None


## Decode Individual Personas

Using the pooled regime parameters, decode each persona's core-dimension trajectory
via the smoothed state probabilities (P(state_t | all data)).

In [ ]:
def decode_dimension(scores_1d: np.ndarray) -> dict:
    """
    Fit a K-state Markov switching model and decode state probabilities.

    Returns
    -------
    dict with keys:
        smoothed_probs : (T, K) smoothed state probabilities
        most_likely    : (T,) most probable state at each step
        alerts         : list of t-indices in state 2 (drifting)
    """
    T = len(scores_1d)
    if T < 4:
        return {"smoothed_probs": np.full((T, K), 1/K),
                "most_likely": np.zeros(T, dtype=int), "alerts": []}
    try:
        m   = MarkovRegression(scores_1d, k_regimes=K, order=0, switching_variance=True)
        fit = m.fit(disp=False, maxiter=200)
        probs = fit.smoothed_marginal_probabilities   # (T, K)
        # Identify which state corresponds to "drifting" (lowest mean)
        means = [fit.params[f"const[{k}]"] for k in range(K)]
        drift_state = int(np.argmin(means))
        most_likely = probs.values.argmax(axis=1)
        alerts = [t for t in range(T) if probs.values[t, drift_state] > 0.6]
        return {
            "smoothed_probs": probs.values,
            "most_likely":    most_likely,
            "drift_state":    drift_state,
            "alerts":         alerts,
            "means":          means,
        }
    except Exception:
        return {"smoothed_probs": np.full((T, K), 1/K),
                "most_likely": np.zeros(T, dtype=int),
                "drift_state": 2, "alerts": [], "means": [0.0, 0.0, 0.0]}


hmm_results = {}

for pid in persona_ids:
    t_idx, matrix = get_persona_matrix(pid)
    T = len(t_idx)
    if T < 5:
        continue

    w = get_profile_weights(pid)
    core_j = core_dim_indices(w, W_MIN)

    dim_results = {}
    alerts = []

    for j in core_j:
        out = decode_dimension(matrix[:, j])
        dim_results[j] = out
        for t in out["alerts"]:
            alerts.append((t, j))

    hmm_results[pid] = {
        "name":        id_to_name.get(pid, pid[:8]),
        "core":        id_to_core.get(pid, []),
        "T":           T,
        "t_idx":       t_idx,
        "matrix":      matrix,
        "weights":     w,
        "dim_results": dim_results,
        "alerts":      alerts,
    }

print(f"HMM decoded {len(hmm_results)} personas\n")
print(f"{'Persona':<25s}  {'Core':<28s}  T  Alerts")
print("-" * 70)
for pid, r in hmm_results.items():
    print(f"{r['name']:<25s}  {', '.join(r['core']):<28s}  {r['T']:2d}  {len(r['alerts'])}")


## Visualise State Probabilities

For each persona × core dimension: alignment scores alongside smoothed state probabilities.
State 2 (drifting) probability in red.

In [ ]:
STATE_COLORS = ["#2ecc71", "#f39c12", "#e74c3c"]  # aligned / struggling / drifting
STATE_LABELS = ["Aligned", "Struggling", "Drifting"]
palette = sns.color_palette("tab10", n_colors=10)


def plot_hmm_persona(pid: str):
    r = hmm_results[pid]
    matrix = r["matrix"]
    T = r["T"]
    core_j = list(r["dim_results"].keys())
    if not core_j:
        return

    fig, axes = plt.subplots(len(core_j), 2, figsize=(14, 2.8 * len(core_j)),
                             sharex=True, gridspec_kw={"width_ratios": [2, 1]})
    if len(core_j) == 1:
        axes = [axes]

    for ax_row, j in zip(axes, core_j):
        ax_score, ax_state = ax_row
        out = r["dim_results"][j]
        scores  = matrix[:, j]
        probs   = out["smoothed_probs"]   # (T, K)
        drift_s = out.get("drift_state", 2)

        ax_score.plot(range(T), scores, "o-", color=palette[j], lw=1.8)
        ax_score.axhline(0, color="grey", lw=0.5, ls="--")
        ax_score.set_ylim(-1.4, 1.4)
        ax_score.set_ylabel(DIM_LABELS[j], fontsize=9)

        for t in out["alerts"]:
            ax_score.axvspan(t - 0.4, t + 0.4, color="red", alpha=0.2, zorder=1)

        # Stacked state probability bars
        bottoms = np.zeros(T)
        for k in range(K):
            ax_state.bar(range(T), probs[:, k], bottom=bottoms,
                         color=STATE_COLORS[k], alpha=0.8, label=STATE_LABELS[k])
            bottoms += probs[:, k]
        ax_state.set_ylim(0, 1)
        ax_state.set_ylabel("P(state)", fontsize=8)

    axes[0][0].set_title(f"{r['name']} — {', '.join(r['core'])}", fontweight="bold")
    axes[0][1].legend(fontsize=7, loc="upper right")
    axes[-1][0].set_xlabel("t_index")
    axes[-1][1].set_xlabel("t_index")
    plt.tight_layout()
    plt.show()


for pid in list(hmm_results.keys())[:4]:
    plot_hmm_persona(pid)


## Limitations

- **Small per-persona data:** With ~10 steps, per-user EM often doesn't converge. Pooled fitting shares parameters but ignores individual variation in transition rates.
- **K is fixed:** K=3 (aligned/struggling/drifting) mirrors the rule-based taxonomy but the true number of regimes is unknown.
- **State labelling:** The 'drifting' state is identified as the one with the lowest mean — but this may not hold if scores are noisy.
- **Profile weights:** HMM operates per-dimension independently. `w_u` must be applied as a post-hoc filter.

HMM becomes the recommended second upgrade path at ~30+ entries per user, where per-user EM becomes stable. See `docs/evolution/drift_detection.md § 3.4`.